# VMirror-SiM Explorer

交互式接口，用于展示、研究和分析车辆拖拽后视镜视野仿真。

**工作流程**：
1. 设置配置目录（默认 `configs/`，可自定义）
2. 通过下拉菜单选择 **车型 / 场景 / 房车 / 后视镜 / 摄像机侧**
3. 运行单元格启动 Blender 并显示

本 notebook **只调用** `src/` 中的接口（`SceneBuilder` / `CameraRig` / `Renderer` / `SimulationPipeline`），不修改其代码。

## 1. 环境准备 — 把项目根加入 `sys.path`

`src` 是项目根下的相对包；从 `notebooks/` 运行时需要先把项目根加入 `sys.path`。

In [ ]:
# Auto-reload src/*.py on every cell execution — 改 src/ 后不用重启 kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## 2. 选择配置目录

默认指向项目下的 `configs/`，如有定制配置（例如 `output/tuned-configs/<session>/`），把它的路径填进 `CONFIG_DIR_OVERRIDE`。

> 注意：`src/_common.py` 里的 `CONFIG_DIR` 是写死的（项目根 / `configs`），所以这里的 "自定义路径" 用于**列举可选项**和**通过 `mirror_path_L/R` 等绝对路径覆盖**——不会改变 `src` 的内置查找路径。

In [ ]:
CONFIG_DIR_OVERRIDE = None   # e.g. PROJECT_ROOT / "output" / "tuned-configs" / "<session>"

CONFIGS_DIR = Path(CONFIG_DIR_OVERRIDE) if CONFIG_DIR_OVERRIDE else (PROJECT_ROOT / "configs")
assert CONFIGS_DIR.exists(), f"Config dir not found: {CONFIGS_DIR}"

def _stems(subdir):
    d = CONFIGS_DIR / subdir
    if not d.exists():
        return []
    return sorted(p.stem for p in d.glob("*.yaml"))

scenes_avail   = _stems("scenes")
vehicles_avail = _stems("vehicles")
caravans_avail = _stems("caravans")

# 后视镜 yaml 都带 _L / _R 后缀；归约成 "型号" 列表
_mirror_L = [s[:-2] for s in _stems("mirrors") if s.endswith("_L")]
_mirror_R = [s[:-2] for s in _stems("mirrors") if s.endswith("_R")]
mirrors_avail = sorted(set(_mirror_L) & set(_mirror_R))

# 摄像机变体（去掉 driver_camera_ 前缀和 _L/_R 后缀）
_cam_stems = [s for s in _stems("cameras") if s.startswith("driver_camera")]
_cam_variants = set()
for s in _cam_stems:
    base = s[len("driver_camera"):]    # "_L" / "_wide_L" / ...
    base = base[:-2] if base.endswith(("_L", "_R")) else base
    base = base.lstrip("_")
    _cam_variants.add(base or "default")
cameras_avail = sorted(_cam_variants)

render_avail = _stems("render")

print("Configs dir   :", CONFIGS_DIR)
print("scenes        :", scenes_avail)
print("vehicles      :", vehicles_avail)
print("caravans      :", caravans_avail, "  (or None)")
print("mirrors       :", mirrors_avail)
print("camera variant:", cameras_avail)
print("render profile:", render_avail)

## 3. 交互选择参数

运行下方单元格后，会出现一组下拉菜单。调整后**直接进入下一节**——无需再运行此单元格。

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import display

def _default(items, preferred):
    return preferred if preferred in items else (items[0] if items else None)

scene_w   = widgets.Dropdown(options=scenes_avail,                    value=_default(scenes_avail,   "lane_change"), description="scene:")
vehicle_w = widgets.Dropdown(options=vehicles_avail,                  value=_default(vehicles_avail, "hilux"),       description="vehicle:")
caravan_w = widgets.Dropdown(options=[None] + caravans_avail,         value=_default(caravans_avail, "large2"),      description="caravan:")
mirror_w  = widgets.Dropdown(options=mirrors_avail,                   value=_default(mirrors_avail,  "standard_convex"), description="mirror:")
side_w    = widgets.Dropdown(options=["L", "R", "both"],              value="both",                                  description="camera side:")
camera_w  = widgets.Dropdown(options=[None] + [c for c in cameras_avail if c != "default"],
                              value=_default(cameras_avail, "wide"),                                                 description="camera variant:")
render_w  = widgets.Dropdown(options=[None] + render_avail,           value=_default(render_avail, "wide"),          description="render profile:")

confirm_btn = widgets.Button(description="确定 ✓", button_style="primary",
                             tooltip="把当前选择打印一遍作为可视反馈；不冻结、不影响下游 cell")
# 用 HTML 而不是 Output —— Output 在 VS Code 里 clear_output 不稳定会累加；
# HTML 是属性替换，每次 .value = 100% 覆盖。
confirm_out = widgets.HTML(value="")

def _on_confirm(_):
    txt = (
        f"scene={scene_w.value}  vehicle={vehicle_w.value}  caravan={caravan_w.value}\n"
        f"mirror={mirror_w.value}  side={side_w.value}\n"
        f"camera_variant={camera_w.value}  render_profile={render_w.value}"
    )
    confirm_out.value = f"<pre style='margin:4px 0;font-family:monospace'>{html.escape(txt)}</pre>"

confirm_btn.on_click(_on_confirm)

ui = widgets.VBox([scene_w, vehicle_w, caravan_w, mirror_w, side_w, camera_w, render_w,
                   confirm_btn, confirm_out])
display(ui)


### 3.b（可选）用绝对路径覆盖

如果想用**调过的 yaml**（比如 `output/tuned-configs/<session>/mirrors/standard_convex_L.yaml`），把绝对路径填到下面对应字段；留空则使用 §3 的下拉菜单结果。

In [ ]:
MIRROR_PATH_L = None    # e.g. str(PROJECT_ROOT / "output/tuned-configs/<session>/mirrors/standard_convex_L.yaml")
MIRROR_PATH_R = None
CAMERA_PATH   = None    # absolute path to a single camera yaml; only valid for side="L" or "R"

MIRROR_TARGET = (0.0, -20.0, 0.5)   # mirror points at (x, y, z) in world space

## 4. 启动 Blender 并显示

三种运行方式（选择其一执行）：

| 模式            | 单元格    | 说明                                                                 |
| --------------- | --------- | -------------------------------------------------------------------- |
| **preview**（推荐展示） | §4.1 | 打开 Blender GUI，进入实时渲染预览（split / single / triple 布局）；可在窗口里调参，反射实时更新。 |
| **render PNG**  | §4.2      | 无窗口模式跑完整 render，把 PNG 写盘并嵌入 notebook 显示。       |
| **build only**  | §4.3      | 只到 SceneBuilder + CameraRig，打开 Blender 让你自己看几何/手动调整。 |

### 4.1 实时预览（GUI + Rendered viewport）

- `layout="split"` — 左半边相机视角（Rendered），右半边自由视角（Solid）
- `layout="triple"` — 左上 L 镜、左下 R 镜、右侧自由视角；**需要 `camera side = both`**
- `layout="single"` — 整个 3D 视口都是相机视角

In [ ]:
from src import SceneBuilder, CameraRig, Renderer

PREVIEW_LAYOUT = "triple"     # "split" | "single" | "triple"

tmp_dir = PROJECT_ROOT / "tmp"
tmp_dir.mkdir(exist_ok=True)
step1 = tmp_dir / "notebook_step1_scene.blend"
step2 = tmp_dir / "notebook_step2_camera.blend"

scene_report = SceneBuilder(
    scene=scene_w.value,
    vehicle=vehicle_w.value,
    caravan=caravan_w.value,
    mirror=mirror_w.value,
    mirror_path_L=MIRROR_PATH_L,
    mirror_path_R=MIRROR_PATH_R,
    mirror_target=MIRROR_TARGET,
).build(output=step1)
print("[scene]", scene_report.get("status"), "→", step1)

camera_report = CameraRig(
    side=side_w.value,
    vehicle=vehicle_w.value,
    camera=camera_w.value,
    camera_path=CAMERA_PATH,
).build(input=step1, output=step2)
print("[camera]", camera_report.get("status"), "→", step2)

_rp = render_w.value
render_profile_arg = f"configs/render/{_rp}.yaml" if _rp else None
preview_handle = Renderer(render_profile=render_profile_arg).preview(
    input=step2,
    layout=PREVIEW_LAYOUT,
)
print("[preview]", preview_handle.get("status"), "  pid=", preview_handle.get("_pid"))
print("  log:", preview_handle.get("_log_path"))

**关闭 Blender 窗口** 用下面这个单元格（或者直接在 GUI 里 `Cmd+Q`）。

In [ ]:
import os, signal
try:
    os.kill(preview_handle["_pid"], signal.SIGTERM)
    print("sent SIGTERM to pid", preview_handle["_pid"])
except (ProcessLookupError, NameError, KeyError) as e:
    print("nothing to close:", e)

### 4.1b 从当前 preview 渲染 PNG（保留 GUI 里的调整）

如果在 §4.1 的 Blender 预览里手动调整了镜面（旋转 / 位移等），运行 §4.2 会**重新跑 SceneBuilder + CameraRig**，从 yaml 重新铺一遍参数，你的实时调整会丢失。

这条路径**跳过 SceneBuilder / CameraRig**——直接对预览用的 blend 跑 `Renderer.render()`，保留镜面的 GUI 调整。

**步骤**：
1. 在 Blender 窗口里调好镜面
2. `Cmd+S` 覆盖保存 `tmp/notebook_step2_camera.blend`（不要"另存为"）
3. 运行下面的 cell（不必先关 Blender）

如果 §3 选了 `camera side = both`，cell 会**自动各渲一遍 L+R**（通过 `Renderer(camera_name=...)` 切换 `scene.camera`），无需在 GUI 里手动改活动相机。

In [ ]:
from src import Renderer
from IPython.display import Image, display

step2 = PROJECT_ROOT / "tmp" / "notebook_step2_camera.blend"
if not step2.exists():
    raise FileNotFoundError(
        f"{step2} 不存在；请先运行 §4.1 生成它（并在 Blender 里 Cmd+S 保存调整）。"
    )

_caravan_tag = caravan_w.value or "nocaravan"
_rp = render_w.value
render_profile_arg = f"configs/render/{_rp}.yaml" if _rp else None

# side='both' 时各渲一遍 L+R；单侧只渲那一台。
sides_to_render = ["L", "R"] if side_w.value == "both" else [side_w.value]

renderer = Renderer(render_profile=render_profile_arg)

for _s in sides_to_render:
    out_png = PROJECT_ROOT / "output" / "render-results" / (
        f"{vehicle_w.value}_{_caravan_tag}_{mirror_w.value}_{_s}_tweaked.png"
    )
    cam_name = f"{vehicle_w.value}_DriverCam_{_s}"

    render_report = renderer.render(
        input=step2,
        output=str(out_png),
        timestamp=True,
        camera_name=cam_name,
    )
    print(f"=== side={_s}  cam={cam_name} ===")
    print("render:", render_report.get("status"))
    rendered = render_report.get("output_png")
    print("PNG   :", rendered)
    if rendered:
        display(Image(filename=rendered))


### 4.2 一键渲染 PNG 并嵌入显示

用 `SimulationPipeline` 一次跑完三步，把 PNG 写到 `output/render-results/`，然后在 notebook 里显示。

In [ ]:
from src import SimulationPipeline
from IPython.display import Image, display

if side_w.value == "both" and CAMERA_PATH is not None:
    raise ValueError(
        "CAMERA_PATH (绝对路径) 只对单侧有效；side='both' 请用 §3 的 camera variant"
    )

_caravan_tag = caravan_w.value or "nocaravan"
_rp = render_w.value
render_profile_arg = f"configs/render/{_rp}.yaml" if _rp else None

# side='both' 就是各渲一遍 L + R；Blender 一次只能渲一台相机的视角。
sides_to_render = ["L", "R"] if side_w.value == "both" else [side_w.value]

for _s in sides_to_render:
    out_png = PROJECT_ROOT / "output" / "render-results" / (
        f"{vehicle_w.value}_{_caravan_tag}_{mirror_w.value}_{_s}.png"
    )

    pipeline_report = SimulationPipeline(
        scene=scene_w.value,
        vehicle=vehicle_w.value,
        caravan=caravan_w.value,
        mirror=mirror_w.value,
        mirror_path_L=MIRROR_PATH_L,
        mirror_path_R=MIRROR_PATH_R,
        mirror_target=MIRROR_TARGET,
        camera_side=_s,
        camera=camera_w.value,
        camera_path=CAMERA_PATH,
        render_profile=render_profile_arg,
        output_png=str(out_png),
        timestamp=True,
    ).run()

    print(f"=== side={_s} ===")
    print("scene :", pipeline_report["scene"].get("status"))
    print("camera:", pipeline_report["camera"].get("status"))
    print("render:", pipeline_report["render"].get("status"))
    rendered = pipeline_report["render"].get("output_png")
    print("PNG   :", rendered)
    if rendered:
        display(Image(filename=rendered))


### 4.3 仅几何（GUI 看场景，不渲染）

适合检查模型放置、镜面方向、房车铰接是否正确——`open_gui=True` 让 Blender 留窗。

In [ ]:
from src import SceneBuilder, CameraRig

tmp_dir = PROJECT_ROOT / "tmp"
tmp_dir.mkdir(exist_ok=True)
step1 = tmp_dir / "notebook_step1_scene.blend"
step2 = tmp_dir / "notebook_step2_camera.blend"

SceneBuilder(
    scene=scene_w.value,
    vehicle=vehicle_w.value,
    caravan=caravan_w.value,
    mirror=mirror_w.value,
    mirror_path_L=MIRROR_PATH_L,
    mirror_path_R=MIRROR_PATH_R,
    mirror_target=MIRROR_TARGET,
).build(output=step1, open_gui=False)

build_only = CameraRig(
    side=side_w.value,
    vehicle=vehicle_w.value,
    camera=camera_w.value,
    camera_path=CAMERA_PATH,
).build(input=step1, output=step2, open_gui=True)

print("[geometry GUI]", build_only.get("status"), "  pid=", build_only.get("_pid"))

## 5. 把 GUI 里调好的参数导出回 yaml（`ConfigExporter`）

如果在 §4.1 的预览里手动调了镜面/相机/房车，并通过 §4.1b 看到效果满意了，可以把这次调整**永久保存**——写回一份 yaml 树到 `output/tuned-configs/<session>/`，下次跑 SceneBuilder 时通过 §2 的 `CONFIG_DIR_OVERRIDE` 指向这个目录就能复用。

**前提**：在 Blender 里 `Cmd+S` 把改动落到 `tmp/notebook_step2_camera.blend`。

**导出的内容**（每项可通过 `include` 选择）：
- `vehicles/<name>.yaml` — origin、eye_point
- `caravans/<name>.yaml` — 位置 + ray_visibility
- `mirrors/<class>_{L,R}.yaml` — glass_center_offset + 显式 rotation
- `cameras/driver_camera{_variant?}_{L,R}.yaml` — lens / sensor / clip

**限制**：相机 yaml 的导出依赖 `vmirror_camera_side`，`side="both"` 时单一相机字段无法表达——所以双侧情况下会自动跳过相机导出（仍会导镜面/车/房车）。

In [ ]:
from src import ConfigExporter

step2 = PROJECT_ROOT / "tmp" / "notebook_step2_camera.blend"
if not step2.exists():
    raise FileNotFoundError(
        f"{step2} 不存在；请先运行 §4.1 → 在 Blender 里调整 → Cmd+S 保存。"
    )

# 留 None 自动生成 session 名（vehicle_caravan_sideX_MMDD_HHMM），
# 或填一个固定字符串覆盖。
EXPORT_TAG: str | None = None

# side='both' 时跳过相机导出（probe 需要单一 camera_side）
include = ("vehicle", "mirror", "caravan", "camera")
if side_w.value == "both":
    include = ("vehicle", "mirror", "caravan")
    print("[info] side='both' → 跳过 camera yaml 导出")

report = ConfigExporter().export(blend=step2, tag=EXPORT_TAG, include=include)

print("session dir:", report["session_dir"])
print("metadata   :", report["metadata"])
print("files written:")
for f in report["files"]:
    print("  -", f)

print("\n下次想用这套参数：把 §2 的 CONFIG_DIR_OVERRIDE 指向上面的 session_dir 即可。")


## 6. 清空 `tmp/`（手动维护用）

每次跑 §4.x / §5 都会在 `tmp/` 下产生 `<feature>_<rand>/` 子目录残留（report.json 之类）；`step1`/`step2` 也是覆盖式的旧版本。

运行下面的 cell 一键清理。注意：清理后再跑 §4.1b 之前**必须先重跑 §4.1**（重新生成 step2.blend）。

In [ ]:
import shutil

tmp_dir = PROJECT_ROOT / "tmp"
if not tmp_dir.exists():
    print(f"{tmp_dir} 不存在，nothing to clean.")
else:
    removed_files = 0
    removed_dirs = 0
    for entry in tmp_dir.iterdir():
        if entry.name in (".gitkeep", ".DS_Store"):
            continue
        if entry.is_dir():
            shutil.rmtree(entry)
            removed_dirs += 1
        else:
            entry.unlink()
            removed_files += 1
    print(f"cleaned {tmp_dir}: removed {removed_dirs} dir(s), {removed_files} file(s)")
